# Preparing EUR dataset for meta-analysis

- Project: RAB GTPase gene investigation and burden analysis across populations with Parkinson’s disease
- Version: Python/3.10.19
- Last Updated: 10-OCT-2026

# Notebook Overview
- 1 Prepare EUR data for RAB29 meta-analysis - Exclude related samples and known variant carriers
  - 1.1 Loading Python libraries
  - 1.2 Set Paths
  - 1.3 Defining functions
  - 1.4 Prepare dataset for meta-analysis - Exclude related individuals from EUR

# Prepare EUR data for *RAB29* meta-analysis - Exclude related samples and known variant carriers

## Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os

# Bring in Pandas for Dataframe functionality
import pandas as pd

# Numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

# Use pathlib for file path manipulation
import pathlib
from pathlib import Path

# Import Sys
import sys as sys

# Import Seaborn
import seaborn as sns

# Install plotnine: a ggplot2-compatible Python plotting package
from plotnine import *

# Import Glob
import glob

# Import Re
import re

# Import Subprocess
import subprocess

# Import CYVCF2
#from cyvcf2 import VCF

# Import tools for meta-analysis
from scipy import stats
from scipy.stats import fisher_exact

# Import collections
from collections import defaultdict, deque

# Import Tempfile
import tempfile

# Import Shutil
import shutil

# Always show all columns in a Pandas DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('future.no_silent_downcasting', True)

## Set paths

In [ ]:
## GP2 v12.0
## Explicitly define release v11.0 path 
REL12_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release12')

WGS_DATA_PATH = pathlib.Path(REL12_PATH, 'wgs/dragen_joint_calling/plink')
WGS_RELATED_PATH = pathlib.Path(REL12_PATH, 'wgs/dragen_joint_calling/related_samples')
CLINICAL_DATA_PATH = pathlib.Path(REL12_PATH, 'clinical_data/master_key_release12_final_vwb.csv')

## Defining functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))


## Prepare dataset for meta-analysis - Exclude related individuals from EUR

In [ ]:
# Find all related samples and exclude all but one from each family (EUR only)

# Merge relatedness per ancestry and stack across ancestries
related_path_1 = "/home/jupyter/workspace/gp2_tier2_eu_release12/wgs/dragen_joint_calling/related_samples"
related_path_2 = "/home/jupyter/workspace/gp2_tier2_eu_release12/meta_data/related_samples"

anc = "EUR"
anc_files = []

# File 1
f1 = os.path.join(related_path_1, f"{anc}/{anc}_release12.related")
if os.path.exists(f1):
    df1 = pd.read_csv(f1, sep=r"[,\t ]+", engine="python")
    df1["Ancestry"] = anc
    anc_files.append(df1)

# File 2
f2 = os.path.join(related_path_2, f"{anc}_release12_vwb.related")
if os.path.exists(f2):
    df2 = pd.read_csv(f2, sep=r"[,\t ]+", engine="python")
    df2["Ancestry"] = anc
    anc_files.append(df2)
    print(f"Found {f2}")

# Merge
if not anc_files:
    raise ValueError("No relatedness files found for EUR!")

rel_df = pd.concat(anc_files, ignore_index=True)
print(f"\nTotal related pairs for EUR: {len(rel_df)}")

In [ ]:
# Build adjacency list from related pairs in EUR only
adj = defaultdict(set)

for _, row in rel_df.iterrows():
    i1 = str(row["IID1"])
    i2 = str(row["IID2"])
    adj[i1].add(i2)
    adj[i2].add(i1)

# Find connected components (families)
visited = set()
components = []

for node in adj.keys():
    if node in visited:
        continue
    comp = []
    q = deque([node])
    visited.add(node)
    while q:
        u = q.popleft()
        comp.append(u)
        for v in adj[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
    components.append(comp)

print(f"Found {len(components)} related clusters")

# Choose one IID per cluster to KEEP, all others EXCLUDED
# Here: keep the first IID in each component
keep_iids = set()
exclude_iids = set()

for comp in components:
    if not comp:
        continue
    keep = comp[0]
    keep_iids.add(keep)
    # Everything else in the family is excluded
    for iid in comp[1:]:
        exclude_iids.add(iid)

print(f"Total related IIDs: {len(keep_iids) + len(exclude_iids)}")
print(f"Keeping {len(keep_iids)}; excluding {len(exclude_iids)}")

In [ ]:
# Exclude variant carriers of causative/pathogenic PD variants from dataset

clinical_data_df_pheno = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)

with open("/home/jupyter/results/RAB29/Meta-analysis/GP2_variant_carriers.txt", "r") as f:
    exclude_ids = [line.strip() for line in f if line.strip()]

# Remove variant carriers from the dataframe
clinical_data_df_pheno_filtered = clinical_data_df_pheno[
    ~clinical_data_df_pheno["GP2ID"].isin(exclude_ids)
].copy()

# Check how many individuals were removed
print(f"Original number of individuals: {len(clinical_data_df_pheno)}")
print(f"Number of variant carriers excluded: {clinical_data_df_pheno['GP2ID'].isin(exclude_ids).sum()}")
print(f"Remaining individuals: {len(clinical_data_df_pheno_filtered)}")

found_ids = set(exclude_ids) & set(clinical_data_df_pheno["GP2ID"])
missing_ids = set(exclude_ids) - set(clinical_data_df_pheno["GP2ID"])

print(f"IDs in carrier file: {len(exclude_ids)}")
print(f"IDs found in dataframe: {len(found_ids)}")
print(f"IDs not found in dataframe: {len(missing_ids)}")

In [ ]:
# Determine number of EUR PD, Controls, and Others with both WGS and genotyping data
clinical_data_df_pheno_EUR = clinical_data_df_pheno_filtered[
    (clinical_data_df_pheno_filtered["nba_label"] == "EUR") |
    (clinical_data_df_pheno_filtered["wgs_label"] == "EUR")
].copy()

# Define conditions for group classification
cond_PD_1 = (clinical_data_df_pheno_EUR['GP2_phenotype_for_qc'] == 'PD') & (clinical_data_df_pheno_EUR['GP2_phenotype'] == 'PD')
cond_PD_2 = ((clinical_data_df_pheno_EUR['GP2_phenotype_for_qc'] == 'Other') &
    ((clinical_data_df_pheno_EUR['GP2_PHENO'] == 'PD') | (clinical_data_df_pheno_EUR['GP2_PHENO'] == 'Affected_PD')) &
    (clinical_data_df_pheno_EUR['diagnosis'] != 'SWEDD') &
    (clinical_data_df_pheno_EUR['diagnosis'] != 'tremor')
)

cond_Con_1 = (clinical_data_df_pheno_EUR['GP2_phenotype_for_qc'] == 'Control')

# Assign group labels
clinical_data_df_pheno_EUR['group'] = 'Others'
clinical_data_df_pheno_EUR.loc[cond_PD_1 | cond_PD_2, 'group'] = 'PD'
clinical_data_df_pheno_EUR.loc[cond_Con_1, 'group'] = 'Control'

# Filter individuals with both wgs_label and nba_label present
clinical_data_df_pheno_EUR_with_both_labels = clinical_data_df_pheno_EUR[clinical_data_df_pheno_EUR['wgs_label'].notna() & clinical_data_df_pheno_EUR['nba_label'].notna()]

# Count how many of each group have both labels
counts_with_both = clinical_data_df_pheno_EUR_with_both_labels['group'].value_counts()

# Filter individuals with both wgs and nba present
clinical_data_df_pheno_EUR_with_both_sequencing = clinical_data_df_pheno_EUR[(clinical_data_df_pheno_EUR['wgs'] == 1) & (clinical_data_df_pheno_EUR['nba'] == 1)]


# Count how many of each group have both labels
counts_with_both2 = clinical_data_df_pheno_EUR_with_both_sequencing['group'].value_counts()

# Show results
print("Number of individuals with both wgs_label and nba_label per group:")
print(counts_with_both)
print("Number of individuals with both wgs and nba per group:")
print(counts_with_both2)


In [ ]:
# Write excluded IDs (EUR only)
exclude_file = "/home/jupyter/results/RAB29/Meta-analysis/Related_excluded_EUR_samples.txt"
with open(exclude_file, "w") as f:
    for iid in sorted(exclude_iids):
        f.write(f"{iid}\n")

excluded_clinical = clinical_data_df_pheno_EUR[
    clinical_data_df_pheno_EUR["GP2ID"].astype(str).isin(exclude_iids)
].copy()

print(f"Excluded samples with clinical data: {excluded_clinical.shape[0]}")

excluded_clinical_file = "/home/jupyter/results/RAB29/Meta-analysis/Related_excluded_EUR_samples_clinical.txt"
excluded_clinical.to_csv(excluded_clinical_file, sep="\t", index=False)


In [ ]:
# Determine number of EUR PD, Controls, and Others after excluding related samples

# Exclude related/clustered samples
clinical_data_df_pheno_excluded = clinical_data_df_pheno_EUR[~clinical_data_df_pheno_EUR["GP2ID"].astype(str).isin(exclude_iids)].copy()


print(f"Total samples before relatedness exclusion: "
      f"{len(clinical_data_df_pheno_EUR)}")

print(f"Related/clustered samples excluded: "
      f"{len(clinical_data_df_pheno_EUR) - len(clinical_data_df_pheno_excluded)}")

print(f"Total samples after relatedness exclusion: "
      f"{len(clinical_data_df_pheno_excluded)}")

# Select EUR individuals with WGS OR NBA data
clinical_data_df_pheno_final = clinical_data_df_pheno_excluded[
    ((clinical_data_df_pheno_excluded["nba_label"] == "EUR") |
        (clinical_data_df_pheno_excluded["wgs_label"] == "EUR"))
    &
    ((clinical_data_df_pheno_excluded["nba"] == 1) |
        (clinical_data_df_pheno_excluded["wgs"] == 1))
].copy()

print(f"\nFinal EUR individuals with WGS and/or NBA data: "
      f"{len(clinical_data_df_pheno_final)}")

# Define phenotype groups
cond_PD_1 = ((clinical_data_df_pheno_final["GP2_phenotype_for_qc"] == "PD") & (clinical_data_df_pheno_final["GP2_phenotype"] == "PD"))
cond_PD_2 = ((clinical_data_df_pheno_final["GP2_phenotype_for_qc"] == "Other") & ((clinical_data_df_pheno_final["GP2_PHENO"] == "PD") |
        (clinical_data_df_pheno_final["GP2_PHENO"] == "Affected_PD")) & (clinical_data_df_pheno_final["diagnosis"] != "SWEDD") & (clinical_data_df_pheno_final["diagnosis"] != "tremor"))

cond_Con_1 = (clinical_data_df_pheno_final["GP2_phenotype_for_qc"] == "Control")

clinical_data_df_pheno_final["group"] = "Others"
clinical_data_df_pheno_final.loc[cond_PD_1 | cond_PD_2, "group"] = "PD"
clinical_data_df_pheno_final.loc[cond_Con_1, "group"] = "Control"


# Define sequencing/data availability
clinical_data_df_pheno_final["data_type"] = "Neither"
clinical_data_df_pheno_final.loc[(clinical_data_df_pheno_final["wgs"] == 1) & (clinical_data_df_pheno_final["nba"] != 1), "data_type"] = "WGS only"
clinical_data_df_pheno_final.loc[(clinical_data_df_pheno_final["wgs"] != 1) & (clinical_data_df_pheno_final["nba"] == 1), "data_type"] = "NBA only"
clinical_data_df_pheno_final.loc[(clinical_data_df_pheno_final["wgs"] == 1) & (clinical_data_df_pheno_final["nba"] == 1), "data_type"] = "WGS + NBA"


# Count final individuals by phenotype
print("\nFinal EUR individuals by phenotype:")
print(clinical_data_df_pheno_final["group"].value_counts())

output_path = "/home/jupyter/results/RAB29/Meta-analysis/Clinical_excluded_related_variant-carrier_samples.txt"
clinical_data_df_pheno_final.to_csv(output_path, sep="\t", index=False)